In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
def prepare_cnn_bilstm_data(file_path):
    df = pd.read_csv(file_path)

    # Parser for Ticker (e.g., AARTIIND28NOV24490PE.NFO)
    def parse_ticker(ticker):
        # Extracts Strike and Expiry String
        match = re.search(r"(\d{2}[A-Z]{3}\d{2})(\d+)[CP]E", str(ticker))
        return match.groups() if match else (None, None)

    # Apply the parser and then assign to multiple columns
    parsed_data = df['Ticker'].apply(parse_ticker)
    df[['Expiry_Str', 'Strike']] = pd.DataFrame(parsed_data.tolist(), index=df.index)

    df = df.dropna(subset=['Strike', 'Close'])
    df['Strike'] = df['Strike'].astype(float)

    # Calculate Time to Expiry (T) and Moneyness (S/K)
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    df['Expiry_dt'] = pd.to_datetime(df['Expiry_Str'], format='%d%b%y')
    df['T'] = (df['Expiry_dt'] - df['Date_dt']).dt.days / 365.0
    df['S_K'] = df['Open'] / df['Strike']

    # Filter valid data
    df = df[df['T'] >= 0]

    # Features: Moneyness, Time, Volume, Open Interest
    features = ['S_K', 'T', 'Volume', 'Open Interest']
    X = df[features].values
    y = (df['Close'] / df['Strike']).values # Normalized Price

    # Scaling (Min-Max as per paper)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Reshape for CNN-BiLSTM: (samples, time_steps, features)
    # Using time_steps=1 for this specific implementation logic
    X_reshaped = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

    return train_test_split(X_reshaped, y, test_size=0.2, random_state=42)

In [ ]:
def build_cnn_bilstm(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # 2 Conv1D layers (64 filters, kernel size 3) as per paper architecture
        # Use padding='same' since time_steps=1
        layers.Conv1D(filters=64, kernel_size=1, activation='relu', padding='same'),
        layers.Conv1D(filters=64, kernel_size=1, activation='relu', padding='same'),

        # 256 BiLSTM units
        layers.Bidirectional(layers.LSTM(256, return_sequences=False)),

        layers.Dropout(0.1),
        layers.Dense(1, activation='linear')
    ])

    # Nadam optimizer with specific learning rate
    optimizer = tf.keras.optimizers.Nadam(learning_rate=0.0001)
    model.compile(optimizer=optimizer, loss='mse')
    return model

In [ ]:
X_train, X_test, y_train, y_test = prepare_cnn_bilstm_data('NSE_FNO_DATA_2024-10-21.CSV')
model = build_cnn_bilstm(X_train.shape[1:])

# Training: 100 epochs, batch size 128 as specified in the paper
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train, y_train, epochs=10, batch_size=256, validation_split=0.2, callbacks=[early_stop], verbose=0)

# Predictions
y_pred = model.predict(X_test).flatten()

# Metrics: RMSE, MAE, R2
mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred)**2)
rmse = np.sqrt(mse)
r2 = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2))

print(f"--- CNN-BiLSTM RESULTS (NSE DATASET) ---")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.4f}")

7459/7459 ━━━━━━━━━━━━━━━━━━━━ 33s 4ms/step
--- CNN-BiLSTM RESULTS (NSE DATASET) ---
MAE:  0.000230
RMSE: 0.000658
R2:   0.9984
